# Azure AI Search ជាមួយ NVIDIA NIM និង LlamaIndex ការរួមបញ្ចូល

នៅក្នុង notebook នេះ យើងនឹងបង្ហាញពីរបៀបប្រើម៉ូដែល AI របស់ NVIDIA និង LlamaIndex ដើម្បីបង្កើតបណ្តាញបច្ចេកវិទ្យា Retrieval-Augmented Generation (RAG) ដែលមានសមត្ថភាពខ្លាំង។ យើងនឹងប្រើ LLMs និង embeddings របស់ NVIDIA, រួមបញ្ចូលពួកវាជាមួយ Azure AI Search ជាខ្ទង់ផ្ទុកវ៉ិចទ័រ (vector store), ហើយធ្វើ RAG ដើម្បីបង្កើនគុណភាពនិងប្រសិទ្ធភាពនៃការស្វែងរក។

## អត្ថប្រយោជន៍
- **សមត្ថភាពពង្រីក**: ប្រើម៉ូដែលភាសាធំៗរបស់ NVIDIA និង Azure AI Search សម្រាប់ការយកវិញដែលអាចពង្រីក និងមានប្រសិទ្ធភាព។
- **ប្រសិទ្ធភាពចំណាយ**: ធ្វើអោយការស្វែងរក និងការយកមកវិញមានប្រសិទ្ធភាពដោយប្រើការផ្ទុកវ៉ិចទ័រដែលមានប្រសិទ្ធភាព និងបច្ចេកទេសស្វែងរកប្រភេទលាយ (hybrid search)។
- **ប្រសិទ្ធភាពខ្ពស់**: រួមបញ្ចូល LLMs ដែលមានអំណាចជាមួយការស្វែងរកចលនា​វ៉ិចទ័រ សម្រាប់ចម្លើយរហ័ស និងត្រឹមត្រូវជាងមុន។
- **គុណភាព**: រក្សាគុណភាពការស្វែងរកខ្ពស់ដោយផ្អែកលើចម្លើយរបស់ LLM ជាមួយឯកសារទាក់ទងដែលបានយកមកវិញ។

## តម្រូវការ
- 🐍 Python 3.9 ឬខ្ពស់ជាងនេះ
- 🔗 [Azure AI Search Service](https://learn.microsoft.com/azure/search/)
- 🔗 កូនសោ API របស់ NVIDIA សម្រាប់ចូលប្រើ LLMs និង Embeddings របស់ NVIDIA តាមរយៈ NVIDIA NIM microservices

## លក្ខណៈពិសេសដែលគ្របដណ្តប់
- ✅ ការរួមបញ្ចូល NVIDIA LLM (យើងនឹងប្រើ [Phi-3.5-MOE](https://build.nvidia.com/microsoft/phi-3_5-moe))
- ✅ NVIDIA Embeddings (យើងនឹងប្រើ [nv-embedqa-e5-v5](https://build.nvidia.com/nvidia/nv-embedqa-e5-v5))
- ✅ របៀបយកមកវិញខ្ពស់របស់ Azure AI Search
- ✅ ការបង្កើតសន្ទស្សន៍ឯកសារជាមួយ LlamaIndex
- ✅ RAG ដោយប្រើ Azure AI Search និង LlamaIndex ជាមួយ NVIDIA LLMs

ចាប់ផ្តើម!


In [ ]:
!pip install azure-search-documents==11.5.1
!pip install --upgrade llama-index
!pip install --upgrade llama-index-core
!pip install --upgrade llama-index-readers-file
!pip install --upgrade llama-index-llms-nvidia
!pip install --upgrade llama-index-embeddings-nvidia
!pip install --upgrade llama-index-postprocessor-nvidia-rerank
!pip install --upgrade llama-index-vector-stores-azureaisearch
!pip install python-dotenv

## ការដំឡើង និងតម្រូវការ
បង្កើតបរិយាកាសសម្រាប់ Python ដែលប្រើកំណែ >3.10.

## ចាប់ផ្តើម!


ដើម្បីចាប់ផ្តើម អ្នកត្រូវការ `NVIDIA_API_KEY` ដើម្បីប្រើម៉ូឌែល NVIDIA AI Foundation:
1) បង្កើតគណនីឥតគិតថ្លៃជាមួយ [NVIDIA](https://build.nvidia.com/explore/discover).
2) ចុចលើម៉ូឌែលដែលអ្នកជ្រើស.
3) នៅក្រោម Input ជ្រើសផ្ទាំង Python ហើយចុច **ទទួល API Key** បន្ទាប់មកចុច **បង្កើត Key**.
4) ចម្លង និងរក្សាទុកកូនសោដែលបានបង្កើតជា NVIDIA_API_KEY. ពីទីនោះ អ្នកគួរតែអាចចូលដល់ endpoints.


In [3]:
import getpass
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvidia_api_key.startswith("nvapi-"), f"{nvidia_api_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key


## ឧទាហរណ៍ RAG ដែលប្រើ LLM និង Embedding
### 1) ចាប់ផ្ដើម LLM
`llama-index-llms-nvidia`, ដែលស្គាល់ថាជារ​ឧបករណ៍ភ្ជាប់ LLM របស់ NVIDIA, អនុញ្ញាតឱ្យអ្នកភ្ជាប់ និងបង្កើតពីម៉ូដែលដែលសមស្រប ដែលមាននៅលើ NVIDIA API catalog។ មើលទីនេះសម្រាប់បញ្ជីម៉ូដែលបញ្ចប់ជាសន្ទនា: https://build.nvidia.com/search?term=Text-to-Text

នៅទីនេះយើងនឹងប្រើ **mixtral-8x7b-instruct-v0.1**


In [75]:
from llama_index.core import Settings
from llama_index.llms.nvidia import NVIDIA

# Here we are using mixtral-8x7b-instruct-v0.1 model from API Catalog
Settings.llm = NVIDIA(model="microsoft/phi-3.5-moe-instruct", api_key=os.getenv("NVIDIA_API_KEY"))

### 2) ចាប់ផ្តើមការបង្កើត Embedding
`llama-index-embeddings-nvidia`, ដែលគេស្គាល់ថា​ជា​ខ្សែភ្ជាប់ Embeddings របស់ NVIDIA, អនុញ្ញាតឱ្យអ្នកភ្ជាប់ និងបង្កើតពីម៉ូដែល​ដែលអាចផ្គូផ្គងបានដែលមាននៅក្នុងបញ្ជី API របស់ NVIDIA។ យើងបានជ្រើស `nvidia/nv-embedqa-e5-v5` ជាម៉ូដែល embedding។ មើលទីនេះសម្រាប់បញ្ជីម៉ូដែល embeddings សម្រាប់អត្ថបទ: https://build.nvidia.com/nim?filters=usecase%3Ausecase_text_to_embedding%2Cusecase%3Ausecase_image_to_embedding


In [6]:
from llama_index.embeddings.nvidia import NVIDIAEmbedding

Settings.embed_model = NVIDIAEmbedding(model="nvidia/nv-embedqa-e5-v5", api_key=os.getenv("NVIDIA_API_KEY"))

### 3) បង្កើតឃ្លាំងវិចទ័រ Azure AI Search


In [76]:
import logging
import sys
import os
import getpass
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from IPython.display import Markdown, display
from llama_index.vector_stores.azureaisearch import AzureAISearchVectorStore, IndexManagement


search_service_api_key = os.getenv('AZURE_SEARCH_ADMIN_KEY') or getpass.getpass('Enter your Azure Search API key: ')
search_service_endpoint = os.getenv('AZURE_SEARCH_SERVICE_ENDPOINT') or getpass.getpass('Enter your Azure Search service endpoint: ')
search_service_api_version = "2024-07-01"
credential = AzureKeyCredential(search_service_api_key)

# Index name to use
index_name = "llamaindex-nvidia-azureaisearch-demo"

# Use index client to demonstrate creating an index
index_client = SearchIndexClient(
    endpoint=search_service_endpoint,
    credential=credential,
)

# Use search client to demonstrate using existing index
search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name=index_name,
    credential=credential,
)

In [ ]:
vector_store = AzureAISearchVectorStore(
    search_or_index_client=index_client,
    index_name=index_name,
    index_management=IndexManagement.CREATE_IF_NOT_EXISTS,
    id_field_key="id",
    chunk_field_key="chunk",
    embedding_field_key="embedding",
    embedding_dimensionality=1024, # dimensionality for nv-embedqa-e5-v5 model
    metadata_string_field_key="metadata",
    doc_id_field_key="doc_id",
    language_analyzer="en.lucene",
    vector_algorithm_type="exhaustiveKnn",
    # compression_type="binary" # Option to use "scalar" or "binary". NOTE: compression is only supported for HNSW
)

### 4) ផ្ទុក, បំបែក, និងអាប់ឡូដ ឯកសារ


In [20]:
from llama_index.core import SimpleDirectoryReader, StorageContext, VectorStoreIndex
from llama_index.core.text_splitter import TokenTextSplitter

# Configure text splitter (nv-embedqa-e5-v5 model has a limit of 512 tokens per input size)
text_splitter = TokenTextSplitter(separator=" ", chunk_size=500, chunk_overlap=10)

# Load documents
documents = SimpleDirectoryReader(
    input_files=["data/txt/state_of_the_union.txt"]
).load_data()
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Create index with text splitter
index = VectorStoreIndex.from_documents(
    documents,
    transformations=[text_splitter],
    storage_context=storage_context,
)

### 5) បង្កើតម៉ាស៊ីនស្វែងរកសំណួរ ដើម្បីសួរព័ត៌មានលើទិន្នន័យរបស់អ្នក

នេះជាការស្នើរសុំមួយដែលប្រើការស្វែងរកវ៉ិចទ័រដោយសុទ្ធក្នុង Azure AI Search និងយោងចម្លើយទៅកាន់ LLM របស់យើង (Phi-3.5-MOE)


In [69]:
query_engine = index.as_query_engine()
response = query_engine.query("Who did the speaker mention as being present in the chamber?")
display(Markdown(f"{response}"))

 The speaker mentioned the Ukrainian Ambassador to the United States, along with other members of Congress, the Cabinet, and various officials such as the Vice President, the First Lady, and the Second Gentleman, as being present in the chamber.

នេះជាសំណើស្វែងរកដែលប្រើការស្វែងរកចម្រុះ (hybrid search) ក្នុង Azure AI Search។


In [70]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.vector_stores.types import VectorStoreQueryMode
from IPython.display import Markdown, display
from llama_index.core.schema import MetadataMode

# Initialize hybrid retriever and query engine
hybrid_retriever = index.as_retriever(vector_store_query_mode=VectorStoreQueryMode.HYBRID)
hybrid_query_engine = RetrieverQueryEngine(retriever=hybrid_retriever)

# Query execution
query = "What were the exact economic consequences mentioned in relation to Russia's stock market?"
response = hybrid_query_engine.query(query)

# Display the response
display(Markdown(f"{response}"))
print("\n")

# Print the source nodes
print("Source Nodes:")
for node in response.source_nodes:
    print(node.get_content(metadata_mode=MetadataMode.LLM))

 The Russian stock market experienced a significant drop, losing 40% of its value. Additionally, trading had to be suspended due to the ongoing situation.



Source Nodes:
file_path: data\txt\state_of_the_union.txt

building a coalition of other freedom-loving nations from Europe and the Americas to Asia and Africa to confront Putin. 

I spent countless hours unifying our European allies. We shared with the world in advance what we knew Putin was planning and precisely how he would try to falsely justify his aggression.  

We countered Russia’s lies with truth.   

And now that he has acted the free world is holding him accountable. 

Along with twenty-seven members of the European Union including France, Germany, Italy, as well as countries like the United Kingdom, Canada, Japan, Korea, Australia, New Zealand, and many others, even Switzerland. 

We are inflicting pain on Russia and supporting the people of Ukraine. Putin is now isolated from the world more than ever. 

Together with our allies –we are right now enforcing powerful economic sanctions. 

We are cutting off Russia’s largest banks from the international financial system.  



#### វិភាគស្វែងរកវ៉ិចទ័រ
ចម្លើយរបស់ LLM បានចាប់យកយ៉ាងត្រឹមត្រូវចំពោះលទ្ធផលសេដ្ឋកិច្ចសំខាន់ៗដែលបានរៀបរាប់នៅក្នុងអត្ថបទប្រភពទាក់ទងនឹងផ្សារហ៊ុនរុស្ស៊ី។ ជាក់លាក់ វាបាននិយាយថា ផ្សារហ៊ុនរុស្ស៊ីបានប្រឈមមុខនឹងការធ្លាក់យ៉ាងខ្លាំង បាត់បង់ 40% នៃតម្លៃរបស់វា ហើយការជួញដូរត្រូវបានផ្អាក ដោយសារស្ថានភាពកំពុងបន្ត។ ចម្លើយនេះសមស្របជាមួយព័ត៌មានដែលបានផ្តល់នៅក្នុងប្រភព បង្ហាញថា LLM បានសម្គាល់ និងសង្ខេបព័ត៌មានដែលពាក់ព័ន្ធខ្លឹមសារ​អំពីផលប៉ៈពាល់លើផ្សារហ៊ុនដោយសារជំហរ​របស់រុស្ស៊ី និងការដាក់ទណ្ឌកម្មដែលបានអនុវត្ត។

#### មតិយោបល់អំពីចំណុចប្រភព
ចំណុចក្នុងប្រភពបានផ្តល់ការរាយការណ៍លម្អិតអំពីលទ្ធផលសេដ្ឋកិច្ចដែលរុស្ស៊ីបានប្រឈមមុខដោយសារការដាក់ទណ្ឌកម្មអន្តរជាតិ។ អត្ថបទបានលើកឡើងថា ផ្សារហ៊ុនរុស្ស៊ីបានបាត់បង់ 40% នៃតម្លៃរបស់វា ហើយការជួញដូរត្រូវបានផ្អាក។ បន្ថែមទៅទៀត វាក៏បានចុះផ្សេងទៀតដល់ផលវិបាកសេដ្ឋកិច្ចផ្សេងៗ ដូចជា ការធ្លាក់តម្លៃនៃរ៉ូបល និងការផ្តាច់ដោយផ្តាច់សេដ្ឋកិច្ចរុស្ស៊ីពីសហគមន៍អន្តរជាតិ។ ចម្លើយរបស់ LLM បានស្តារចំណុចសំខាន់ៗពីចំណុចទាំងនេះយ៉ាងមានប្រសិទ្ធភាព ដោយផ្តោតទៅលើផលប៉ៈពាល់លើផ្សារហ៊ុនដូចដែលត្រូវបានស្នើក្នុងសំនើរ។


ឥឡូវនេះ, យើងមកមើលសំណួរមួយដែល Hybrid Search មិនបានផ្តល់ចម្លើយដែលមានមូលដ្ឋានល្អ៖


In [71]:
# Query execution
query = "What was the precise date when Russia invaded Ukraine?"
response = hybrid_query_engine.query(query)

# Display the response
display(Markdown(f"{response}"))
print("\n")

# Print the source nodes
print("Source Nodes:")
for node in response.source_nodes:
    print(node.get_content(metadata_mode=MetadataMode.LLM))


 The provided context does not specify the exact date of Russia's invasion of Ukraine. However, it does mention that the events discussed are happening in the current era and that the actions taken are in response to Putin's aggression. For the precise date, one would need to refer to external sources or historical records.



Source Nodes:
file_path: data\txt\state_of_the_union.txt

our forces are not engaged and will not engage in conflict with Russian forces in Ukraine.  

Our forces are not going to Europe to fight in Ukraine, but to defend our NATO Allies – in the event that Putin decides to keep moving west.  

For that purpose we’ve mobilized American ground forces, air squadrons, and ship deployments to protect NATO countries including Poland, Romania, Latvia, Lithuania, and Estonia. 

As I have made crystal clear the United States and our Allies will defend every inch of territory of NATO countries with the full force of our collective power.  

And we remain clear-eyed. The Ukrainians are fighting back with pure courage. But the next few days weeks, months, will be hard on them.  

Putin has unleashed violence and chaos.  But while he may make gains on the battlefield – he will pay a continuing high price over the long run. 

And a proud Ukrainian people, who have known 30 years  of independence,

### Hybrid Search: ការវិភាគចម្លើយ LLM
ចម្លើយរបស់ LLM ក្នុងឧទាហរណ៍ Hybrid Search បង្ហាញថាបរិបទដែលបានផ្តល់មិនបានបញ្ជាក់កាលបរិច្ឆេទជាក់លាក់នៃការវាយប្រហាររបស់រុស្ស៊ីទៅលើអ៊ុយក្រែន។ ចម្លើយនេះបញ្ជាក់ថា LLM កំពុងយកព័ត៌មានដែលមានក្នុងឯកសារប្រភពមកប្រើ ប៉ុន្តែទទួលស្គាល់ពីការមិនមានព័ត៌មានលំអិតជាក់លាក់នៅក្នុងអត្ថបទ។

ចម្លើយនេះត្រឹមត្រូវក្នុងការបញ្ជាក់ថាបរិបទបានរៀបរាប់ពីព្រឹត្តិការណ៍ដែលពាក់ព័ន្ធនឹងការវាយប្រហាររបស់រុស្ស៊ី ប៉ុន្តែមិនអាចកំណត់កាលបរិច្ឆេទនៃការវាយប្រហារ​បាន។ នេះបង្ហាញពីសមត្ថភាពរបស់ LLM ក្នុងការយល់ដឹងអំពីព័ត៌មានដែលបានផ្តល់ ខណៈដែលវាស្គាល់ពីចន្លោះក្នុងមាតិកា។ LLM បានលើកទឹកចិត្តឲ្យអ្នកប្រើស្វែងរកប្រភពខាងក្រៅឬកំណត់ត្រាប្រវត្តិសាស្ត្រសម្រាប់កាលបរិច្ឆេទជាក់លាក់ ដោយបង្ហាញពីការប្រុងប្រយ័ត្នពេលព័ត៌មានមិនពេញលេញ។

### ការវិភាគចំណុចប្រភព
ចំណុចប្រភពក្នុងឧទាហរណ៍ Hybrid Search មានអត្ថបទចេញពីសុន្ទរកថាមួយដែលពិភាក្សាអំពីការឆ្លើយតបរបស់សហរដ្ឋអាមេរិកចំពោះសកម្មភាពរបស់រុស្ស៊ីនៅអ៊ុយក្រែន។ ចំណុចទាំងនេះលើកស្ទង់ពីផលប៉ះពាល់យុទ្ធសាស្ត្រទូលំទូលាយ និងជំហានដែលសហរដ្ឋអាមេរិក និងដៃគូរបស់ខ្លួនបានអនុវត្តក្នុងការឆ្លើយតបទៅនឹងការវាយប្រហារ ប៉ុន្តែពួកវាមិនបានរៀបរាប់កាលបរិច្ឆេទជាក់លាក់នៃការវាយប្រហារ។ នេះស្របគ្នានឹងចម្លើយរបស់ LLM ដែលបានកំណត់យ៉ាងត្រឹមត្រូវថាបរិបទខ្វះព័ត៌មានកាលបរិច្ឆេទជាក់លាក់។


In [72]:
# Initialize hybrid retriever and query engine
semantic_reranker_retriever = index.as_retriever(vector_store_query_mode=VectorStoreQueryMode.SEMANTIC_HYBRID)
semantic_reranker_query_engine = RetrieverQueryEngine(retriever=semantic_reranker_retriever)

# Query execution
query = "What was the precise date when Russia invaded Ukraine?"
response = semantic_reranker_query_engine.query(query)

# Display the response
display(Markdown(f"{response}"))
print("\n")

# Print the source nodes
print("Source Nodes:")
for node in response.source_nodes:
    print(node.get_content(metadata_mode=MetadataMode.LLM))


 The provided context does not specify the exact date of Russia's invasion of Ukraine. However, it mentions that the event occurred six days before the speech was given. To determine the precise date, one would need to know the date of the speech.



Source Nodes:
file_path: data\txt\state_of_the_union.txt

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world. 

### Hybrid w/Reranking: LLM Response Analysis
នៅក្នុងឧទាហរណ៍ Hybrid w/Reranking ចំលើយរបស់ LLM ផ្ដល់បរិបទបន្ថែមដោយសម្គាល់ថាព្រឹត្តិការណ៍នេះបានកើតឡើង ៦ ថ្ងៃមុនសុន្ទរកថា ដែលបង្ហាញថា LLM អាចសន្មត់កាលបរិច្ឆេទនៃការវាយប្រហារបានដោយផ្អែកលើពេលវេលានៃសុន្ទរកថា ទោះជាក៏បីជា វានៅតែត្រូវការដឹងកាលបរិច្ឆេទពិតប្រាកដនៃសុន្ទរកថា ដើម្បីមានភាពត្រឹមត្រូវ។

ចំលើយនេះបង្ហាញពីសមត្ថភាពប្រសើរឡើងក្នុងការប្រើសញ្ញាបរិបទ ដើម្បីផ្ដល់ចម្លើយដែលមានព័ត៌មានច្រើនជាងមុន។ វាបង្ហាញអំពីអត្ថប្រយោជន៍នៃ reranking ដែលនៅពេលនោះ LLM អាចចូលដល់ និងផ្តល់អាទិភាពដល់ព័ត៌មានដែលពាក់ព័ន្ធបន្ថែម ដើម្បីផ្ដល់ការសន្មត់ឱ្យនៅជិតនឹងព័ត៌មានដែលត្រូវការពិត (ឧ. កាលបរិច្ឆេទនៃការវាយប្រហារ)។

### Source Nodes Analysis
ចំណុចប្រភពក្នុងឧទាហរណ៍នេះរួមមានយោងទៅលើពេលវេលានៃការវាយប្រហាររបស់រុស្ស៊ី ដោយបានចែងយ៉ាងច្បាស់ថា វាបានកើតឡើង ៦ ថ្ងៃមុនសុន្ទរកថា។ ទោះបីជាកាលបរិច្ឆេទពិតប្រាកដមិនទាន់បានបញ្ជាក់យ៉ាងរឹងបំពាន ក៏ដោយ ចំណុចប្រភពទាំងនេះផ្តល់បរិបទពេលវេលាដែលអនុញ្ញាតឲ្យ LLM ផ្តល់ចម្លើយដែលមានលក្ខណៈលម្អិតបន្ថែម។ ការជួយបញ្ចូលព័ត៌មានលម្អិតនេះបង្ហាញពីរបៀបដែល reranking អាចធ្វើឲ្យសមត្ថភាពរបស់ LLM ក្នុងការដកយក និងសន្មត់ព័ត៌មានពីបរិបទដែលបានផ្ដល់មានប្រសើរឡើង ដោយបណ្តាលឲ្យបានចម្លើយដែលមានភាពត្រឹមត្រូវ និងផ្ដល់ព័ត៌មានច្រើនជាងមុន។


**កត់សំគាល់៖**
នៅក្នុងកំណត់ត្រានេះ យើងបានប្រើមីក្រូសេវា NVIDIA NIM ពីកាតាឡុក API របស់ NVIDIA។
API ខាងលើ `NVIDIA (llms)`, `NVIDIAEmbedding`, និង [Azure AI Search Semantic Hybrid Retrieval (ការស្វែងរកលាយឡំដោយអត្ថន័យ - ការរៀបចំឡើងវិញបញ្ចូលក្នុង)](https://learn.microsoft.com/azure/search/semantic-search-overview). សូមចំណាំថា API ខាងលើទាំងនេះ ក៏អាចគាំទ្រមីក្រូសេវាដែលផ្ទុកដោយខ្លួនឯងបានផងដែរ។ 

**ឧទាហរណ៍៖**
```python
NVIDIA(model="meta/llama3-8b-instruct", base_url="http://your-nim-host-address:8000/v1")


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះបីយើងខិតខំផ្តោតលើភាពត្រឹមត្រូវ សូមចំណាំថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬមិនច្បាស់លាស់បាន។ ឯកសារដើមក្នុងភាសាមានដើមរបស់វាគួរត្រូវបានចាត់ទុកថាជាប្រភពដែលមានសិទ្ធិបំផុត។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឱ្យអនុវត្តការបកប្រែដោយអ្នកជំនាញមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
